In [0]:
from pyspark.sql.functions import current_timestamp

source_location="/Volumes/data_modelling_demo_catalog/source/demo_volume/SCD_1/"
target_location="data_modelling_demo_catalog.bronze.SCD_1_output"
checkpoint_location="/Volumes/data_modelling_demo_catalog/source/demo_volume/checkpoints/SCD_1_output/"
schema_location="/Volumes/data_modelling_demo_catalog/source/demo_volume/schemas/SCD_1_output/"

df= (spark.readStream.\
    #read the data using autoloader batch mode
    format("cloudFiles").\
    option("cloudFiles.format","csv").\
    option("header","true").\
    option("cloudFiles.schemaLocation", schema_location).\
    option("cloudFiles.schemaEvolutionMode","addNewColumns").\
    load(source_location).\
    #perform the transformation
    withColumn("ingestion_time", current_timestamp()).\
    #write the data to a delta table
    writeStream.format("delta").\
    option("mergeSchema", "true").\
    outputMode("append").\
    option("checkpointLocation", checkpoint_location).\
    trigger(availableNow=True).\
    toTable(target_location)
)

df.awaitTermination()
print(f"Operation completed successfully!")


In [0]:
%sql
-- select count(*) from data_modelling_demo_catalog.bronze.scd_1_output;

select * from data_modelling_demo_catalog.bronze.scd_1_output limit 10;